In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [3]:
data=pd.read_csv('Churn_Modelling_copy.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
## estimate salary column will be our regression
# Preprocess the data
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [11]:
# Encode categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [12]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,730,1,42,4,0.00,2,0,1,85982.47,0,0.0,0.0,1.0
95,515,1,35,10,176273.95,1,0,1,121277.78,0,0.0,0.0,1.0
96,773,1,41,9,102827.44,1,0,1,64595.25,0,0.0,0.0,1.0
97,814,1,29,8,97086.40,2,1,1,197276.13,0,0.0,1.0,0.0


In [13]:
# One-hot encode 'Geography'
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

KeyError: "None of [Index(['Geography'], dtype='object')] are in the [columns]"

In [14]:
# Combine one-hot encoded columns with original data
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

KeyError: "['Geography'] not found in axis"

In [15]:
# Split the data into features and target
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']

In [16]:
## Split the data in training and tetsing sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
## Scale these features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [18]:
# Save the encoders and scaler for later use
with open('Label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

### Train our ANN with regression Problem Statement

In [19]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [20]:
# Build the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer for regression
])

## compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [21]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

# Set up TensorBoard
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [22]:
# Set up Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


In [23]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/100
3/3 [==============================] - 0s 68ms/step - loss: 102480.5703 - mae: 102480.5703 - val_loss: 110722.9375 - val_mae: 110722.9375
Epoch 2/100
3/3 [==============================] - 0s 16ms/step - loss: 102480.3438 - mae: 102480.3438 - val_loss: 110722.7109 - val_mae: 110722.7109
Epoch 3/100
3/3 [==============================] - 0s 17ms/step - loss: 102480.1172 - mae: 102480.1172 - val_loss: 110722.4844 - val_mae: 110722.4844
Epoch 4/100
3/3 [==============================] - 0s 14ms/step - loss: 102479.8984 - mae: 102479.8984 - val_loss: 110722.2734 - val_mae: 110722.2734
Epoch 5/100
3/3 [==============================] - 0s 15ms/step - loss: 102479.6797 - mae: 102479.6797 - val_loss: 110722.0625 - val_mae: 110722.0625
Epoch 6/100
3/3 [==============================] - 0s 14ms/step - loss: 102479.4766 - mae: 102479.4766 - val_loss: 110721.8516 - val_mae: 110721.8516
Epoch 7/100
3/3 [==============================] - 0s 15ms/step - loss: 102479.2656 - mae: 102479.26

In [24]:
%load_ext tensorboard

In [25]:
%tensorboard --logdir regressionlogs/fit

In [26]:
## Evaluate model on the test data
test_loss,test_mae=model.evaluate(X_test,y_test)
print(f'Test MAE : {test_mae}')

1/1 [==============================] - 0s 79ms/step - loss: 110453.2891 - mae: 110453.2891
Test MAE : 110453.2890625


In [27]:
model.save('regression_model2.h5')

/Users/harshveersinghnirwan/Downloads/annclassification_myproject/venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
